# Groceries — phân tích giỏ hàng

Giỏ có rau củ, trái cây và sữa có thường thêm một nhóm rau khác không? Khai thác đồng xuất hiện giúp đặt giả thuyết về vị trí trưng bày hoặc gợi ý mua kèm. ECLAT tìm tập sản phẩm phổ biến từ các giỏ. Luật kết hợp đo xem hậu kiện đi cùng tiền kiện với tần suất nào và so với tần suất nền ra sao.

| Thành phần | Dùng để làm gì |
|---|---|
| Python chuẩn (`csv`, `collections`, `json`, `tomllib`), curl | Tải nguồn có SHA-256, đổi mỗi giỏ thành CSV dài, đọc bảng và thống kê mô tả |
| Julia Base và thư viện chuẩn `Unicode`, `TOML`, `SHA` | Chuẩn hóa tên, mã hóa ID, chạy ECLAT và sinh luật bằng mã nhóm |
| IPython và Matplotlib | Hiển thị bảng và phân bố số sản phẩm mỗi giỏ |
| Java và SPMF 2.64b | Đối chiếu map itemset/support sau khai thác |

Nguồn là bộ Groceries được mô tả trong `arules`; CSV tải từ snapshot Git công khai có revision và checksum cố định. Các giỏ tương ứng một tháng quan sát tại điểm bán. Nguồn CSV không ghi rõ điều kiện tái phân phối riêng, nên dùng link và checksum để tải. [Mô tả dữ liệu arules](https://mhahsler.github.io/arules/reference/Groceries.html) · [Nguồn và checksum](../data/manifest.csv)

## Môi trường và thư mục kết quả

Dùng Colab CPU hoặc Jupyter từ một bản clone của repo. Chọn **Run all** từ đầu. Ô khởi tạo in commit, phiên bản môi trường và tạo thư mục kết quả riêng. Ở Colab nó clone public `main`. Trên máy cá nhân cần Julia ≥1.9, Python ≥3.11, Java ≥21 và curl, theo [README](../README.md).

In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    ROOT = Path(tempfile.mkdtemp(prefix="eclat-", dir="/content"))
    URL = "https://github.com/phuctran33/-DM-Eclat.git"
    REF = "main"
    subprocess.run(["git", "clone", "--single-branch", "--branch", REF, URL, str(ROOT)], check=True)
else:
    ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
                 if (path / "src/EclatProject.jl").is_file()), None)
    if ROOT is None:
        raise RuntimeError("Open the notebook from the project checkout.")

sys.path.insert(0, str(ROOT / "notebooks"))
from eclat_setup import csv_rows, run, run_julia, setup, show_table

OUTPUT, JULIA, JAVA = setup(ROOT, IN_COLAB, "01_groceries")

## 1. Đơn vị phân tích và dữ liệu thô

Mỗi dòng CSV nguồn là một lần mua, chỉ chứa các tên mặt hàng. Adapter gán cho từng dòng một basket ID rồi ghi các cặp `basket_id,item_name`. Tên sản phẩm rỗng bị báo lỗi. Kiểm hash trước khi chuẩn hóa giúp phát hiện snapshot thay đổi. Hai giỏ giống nội dung vẫn là hai lần mua.

In [ ]:
from shutil import copyfile
import csv
import tomllib

run(sys.executable, "-B", "scripts/prepare_application_data.py", "--dataset", "groceries")
source_record = OUTPUT / "dataset-source.json"
copyfile(ROOT / "data/applications/groceries_source.json", source_record)
print("Nguồn tải và SHA-256:", source_record)
with (ROOT / ".tools/downloads/groceries-source.csv").open(encoding="utf-8", newline="") as stream:
    raw_baskets = list(csv.reader(stream))
show_table(["Dòng nguồn", "Tên sản phẩm", "Độ rộng thô"],
           [(i, ", ".join(row), len(row)) for i, row in enumerate(raw_baskets[:3], 1)])
canonical = csv_rows(ROOT / "data/applications/groceries.csv")
show_table(["basket_id", "item_name (CSV dài)"], [(r["basket_id"], r["item_name"]) for r in canonical[:6]])
print(f"{len(raw_baskets):,} giỏ nguồn, {len(canonical):,} cặp giỏ–sản phẩm trước mã hóa.")

## 2. Chuẩn hóa, mã hóa và khai thác

Julia chuẩn hóa Unicode NFC, chữ thường, cắt và gộp khoảng trắng. Tên chuẩn được sắp từ điển rồi đánh ID. Mỗi basket ID tạo một giao dịch SPMF, một item xuất hiện tối đa một lần trong giỏ. Lưu bảng ID để giải mã kết quả.

Ngưỡng 0,005 giữ tập xuất hiện ở ít nhất 50 giỏ trên snapshot này. Đây là ngưỡng thăm dò, giữ các kết hợp ít gặp mà vẫn có tối thiểu vài chục quan sát. Confidence tối thiểu 0,60 lọc chiều luật. Thay đổi hai ngưỡng sẽ đổi tập và luật.

Ô dưới chạy pipeline thật, hai backend so đủ map trước khi ghi, rồi SPMF đối chiếu độc lập trên cùng giao dịch. CSV luật và bảng dưới đây đọc từ lần chạy mới.

In [ ]:
from collections import Counter
from IPython.display import Image, display
import matplotlib.pyplot as plt

app = OUTPUT / "applications"
run_julia("scripts/run_applications.jl", "--dataset", "groceries", "--output", app)
run(sys.executable, "-B", "scripts/verify_application_results.py", "--dataset", "groceries",
    "--output", app, "--java", JAVA)

prepared = app / "prepared/groceries"
items = {int(r["item_id"]): r["item_label"] for r in csv_rows(prepared / "items.csv")}
baskets = [set(map(int, line.split()))
           for line in (prepared / "transactions.txt").read_text(encoding="utf-8").splitlines()]
audit = csv_rows(prepared / "preparation_audit.csv")
show_table(["Kiểm tra preprocessing", "Số lượng"], [(r["reason"], r["count"]) for r in audit])
show_table(["TID", "Mã SPMF", "Giải mã sau chuẩn hóa"], [
    (i, " ".join(map(str, sorted(ids))), ", ".join(items[j] for j in sorted(ids)))
    for i, ids in enumerate(baskets[:3], 1)
])
frequencies = Counter(item for basket in baskets for item in basket)
show_table(["Sản phẩm", "Số giỏ", "Tỷ lệ (%)"], [
    (items[item], count, f"{100 * count / len(baskets):.2f}")
    for item, count in frequencies.most_common(8)
])

widths = Counter(map(len, baskets))
fig, ax = plt.subplots(figsize=(7.5, 3.4), layout="constrained")
ax.bar(sorted(widths), [widths[k] for k in sorted(widths)], color="#31708e")
ax.set(title="Phân bố độ rộng giỏ sau chuẩn hóa", xlabel="Số sản phẩm khác nhau", ylabel="Số giỏ")
chart = OUTPUT / "groceries-width.png"
fig.savefig(chart, dpi=140)
plt.close(fig)
display(Image(filename=str(chart)))

meta = tomllib.loads((app / "run.toml").read_text(encoding="utf-8"))
stats = meta["statistics"]["groceries"]
show_table(["Giỏ", "Items", "Minsup", "Min count", "Tập phổ biến", "Luật"],
           [(stats["n_transactions"], stats["n_items"], meta["config"]["groceries"]["minsup"],
             stats["min_count"], stats["n_patterns"], stats["n_rules"])])
sys.path.insert(0, str(ROOT / "scripts"))
from benchlib import read_patterns

patterns = read_patterns(app / "patterns/groceries.txt")
pairs = sorted(((key, n) for key, n in patterns.items() if len(key) == 2),
               key=lambda pair: (-pair[1], pair[0]))[:6]
show_table(["Cặp phổ biến", "Số giỏ"], [(", ".join(items[item] for item in key), n) for key, n in pairs])

Độ rộng đo số sản phẩm khác nhau sau chuẩn hóa. Các giỏ dài có nhiều cơ hội sinh ứng viên hơn. Những sản phẩm nền phổ biến xuất hiện trong nhiều giỏ, nên phải đọc tần suất nền khi diễn giải luật có chúng ở vế phải.

## 3. Từ tập phổ biến đến luật

Với $A$ và $B$ khác rỗng, rời nhau, gọi $n$ là số giỏ và $\sigma$ là count chứa một tập.

- Support $=\sigma(A\cup B)/n$, cho biết luật xuất hiện trên bao nhiêu phần số giỏ.
- Confidence $=\sigma(A\cup B)/\sigma(A)$, là tần suất hậu kiện trong các giỏ có tiền kiện.
- Lift $=\operatorname{confidence}/(\sigma(B)/n)$, so sánh với tỷ lệ nền của hậu kiện.

Hàm Julia xét mọi cách tách tập frequent thành hai vế. Quy tắc xếp hạng giảm dần lift, confidence, support, rồi tăng độ dài và ID. Bảng hiển thị số đã làm tròn để đọc, việc lọc và xếp hạng dùng giá trị gốc.

In [ ]:
top = csv_rows(app / "groceries_top10.csv")
show_table(["Hạng", "Tiền kiện", "Hậu kiện", "Giỏ cả hai vế", "Support (%)", "Confidence (%)", "Lift"], [
    (r["rank"], r["antecedent_labels"].replace(" | ", ", "), r["consequent_labels"].replace(" | ", ", "),
     r["support_count"], f'{100 * float(r["support"]):.4f}', f'{100 * float(r["confidence"]):.2f}', f'{float(r["lift"]):.3f}')
    for r in top
])
for i in (0, 4, 8):
    r = top[i]
    joint = int(r["support_count"])
    lhs, rhs = int(r["antecedent_count"]), int(r["consequent_count"])
    n = int(r["n_transactions"])
    print(f'Hạng {r["rank"]}: support {joint}/{n} = {joint/n:.4%}; confidence {joint}/{lhs} = {joint/lhs:.2%}; '
          f'nền {rhs}/{n} = {rhs/n:.2%}; lift {(joint/lhs)/(rhs/n):.3f}')
antecedents = [set(map(int, r["antecedent_ids"].split())) for r in top]
unions = [a | set(map(int, r["consequent_ids"].split())) for a, r in zip(antecedents, top)]
print("Giỏ khớp ít nhất một tiền kiện:", sum(any(a <= basket for a in antecedents) for basket in baskets))
print("Giỏ khớp ít nhất một luật đủ hai vế:", sum(any(u <= basket for u in unions) for basket in baskets))
print("Bằng chứng:", app / "validation.json")

## 4. Đọc ba luật và ra quyết định

- **Hạng 1, trái cây họ cam, rau củ và sữa → các rau khác.** Hàng đầu có lift cao vì 57/90 giỏ đủ tiền kiện chứa rau khác, cao hơn tỷ lệ nền khoảng 19%. Có thể thử đề xuất rau khác cho đúng tổ hợp. Số giỏ khớp cả hai vế nhỏ, nên cần đo trên giai đoạn khác.
- **Hạng 5, rau củ, trái cây nhiệt đới và yogurt → sữa.** Confidence 70% so với nền sữa khoảng 26% cho thấy gợi ý bán kèm có cơ sở khi phù hợp tiền kiện. Thiết kế thử nghiệm ngẫu nhiên trong nhóm giỏ đủ điều kiện, đo tỉ lệ thêm sữa giữa có và không có gợi ý.
- **Hạng 9, bơ và yogurt → sữa.** Luật có support cao hơn hạng 1 và tiền kiện ngắn hơn, lift thấp hơn. Đây có thể là lựa chọn dễ thử khi ưu tiên phạm vi phục vụ. So tỉ lệ thêm sữa và giá trị giỏ với nhóm đối chứng.

Không cộng support từng luật để tính số giỏ phủ vì các tiền kiện có thể chồng lắp. Tỉ lệ nền cao có thể cho confidence lớn dù lift gần 1. Lift là tỷ số tần suất đồng xuất hiện theo mô hình độc lập và cần một thử nghiệm để đánh giá tác động kinh doanh. Dữ liệu không chứa giá, thời điểm từng giỏ hoặc can thiệp khuyến mãi. [CSV kết quả công bố](../results/applications/groceries_top10.csv)